# Indicador de Propensión a Incendios (IPI) municipal y mensual
**Fuente:** IDEAM - Sistema de Monitoreo de Puntos de Calor (https://puntosdecalor.ideam.gov.co/)

**Flujo manual:** descargar desde el portal el CSV de puntos de calor de Colombia y guardarlo en la carpeta `Output4/Indicadores/Incendios` con el nombre `colombia_<inicio>_<fin>.csv`. El notebook toma el archivo más reciente con ese patrón.

Usa el histórico de focos de calor satelitales (VIIRS) con el **día** como unidad de análisis y construye por municipio un índice compuesto de frecuencia, extensión, intensidad y persistencia del fuego, en versión anual y por mes calendario. Al final exporta `incendios_ultimo_dia.csv` para el dashboard web, con el mismo formato de siempre.

Los archivos finales se escriben en `webpage_climate/data`; las salidas intermedias pesadas quedan en `salidas_ipi`, dentro de la carpeta de datos crudos (fuera del repositorio).

In [ ]:
import subprocess
import time
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- Rutas ---
RAW_DIR = Path(r'C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output4\Indicadores\Incendios')
RAW_PATRON = 'colombia_*.csv'   # descarga manual del portal de puntos de calor
SALIDAS_IPI = RAW_DIR / 'salidas_ipi'

REPO_DIR = Path.cwd().parent  # carpeta padre de scripts_python
origen = subprocess.run(['git', '-C', str(REPO_DIR), 'remote', 'get-url', 'origin'],
                        capture_output=True, text=True).stdout.strip()
assert 'github.com/gadielfelipe/sources_of_0015' in origen, f'Repositorio no encontrado en {REPO_DIR} (origin: {origen!r})'
WEB_DATA = REPO_DIR / 'webpage_climate' / 'data'
WEB_CSV = WEB_DATA / 'incendios_ultimo_dia.csv'

# Archivo crudo más reciente
candidatos = sorted(RAW_DIR.glob(RAW_PATRON), key=lambda p: p.stat().st_mtime)
assert candidatos, f'No se encontró {RAW_PATRON} en {RAW_DIR}'
RAW_FILE = candidatos[-1]
SALIDAS_IPI.mkdir(parents=True, exist_ok=True)

# --- Periodo y filtros ---
FECHA_INI = pd.Timestamp('2024-01-01')
FECHA_FIN = None                          # None = último mes completo del archivo
FUENTES = ['VIIRS-NOAA-20', 'VIIRS-NOAA-21', 'VIIRS-Suomi-NPP']
FUENTE_SENSIBILIDAD = 'VIIRS-Suomi-NPP'
CONFIANZA_MIN = 'Nominal'                 # se descartan los focos de confianza Baja
BBOX = dict(lat_min=-4.3, lat_max=13.5, lon_min=-82.0, lon_max=-66.8)

# --- Fuentes de calor persistentes (teas de gas, industria) ---
CELDA_GRADOS = 0.004
MIN_ANIOS = 3        # años activos mínimos (el periodo solo tiene 2024, 2025 y 2026; el diseño original pedía 4)
MIN_DIAS_ANIO = 8    # mediana mínima de días con detección por año activo
MIN_MESES = 8        # meses calendario distintos mínimos

# --- Índice ---
GAP_EPISODIO = 3     # días sin fuego que separan dos episodios
MIN_DIAS_INTENSIDAD = 5
PESOS = dict(frecuencia=0.40, extension=0.25, intensidad=0.20, persistencia=0.15)
CATEGORIAS = ['MUY_BAJA', 'BAJA', 'MEDIA', 'ALTA', 'MUY_ALTA']

print(f'Repositorio: {REPO_DIR}  ({origen})')
print(f'Datos crudos: {RAW_FILE}')
print(f'Salidas intermedias: {SALIDAS_IPI}')

## Paso 0. Carga e inspección

In [ ]:
COLUMNAS = ['fecha', 'lat', 'lon', 'fuente', 'temp_i4', 'temp_i5', 'frp',
            'confianza', 'dia_noche', 'scan_km', 'track_km']
NUMERICAS = ['lat', 'lon', 'temp_i4', 'temp_i5', 'frp', 'scan_km', 'track_km']


def leer_focos(ruta):
    """Lee el CSV de focos de calor de IDEAM (sep ';', decimal ',') y estandariza columnas y tipos."""
    t0 = time.time()
    for enc in ('utf-8', 'latin-1'):
        try:
            df = pd.read_csv(ruta, sep=';', decimal=',', encoding=enc, dtype=str)
            break
        except UnicodeDecodeError:
            continue
    df = df.loc[:, ~df.columns.str.startswith('Unnamed')]
    assert df.shape[1] == len(COLUMNAS), f'Columnas inesperadas: {df.columns.tolist()}'
    df.columns = COLUMNAS
    for c in NUMERICAS:
        df[c] = pd.to_numeric(df[c].str.replace(',', '.'), errors='coerce')
        if c not in ('lat', 'lon'):  # coordenadas en float64 para no perder precisión
            df[c] = df[c].astype('float32')
    df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')  # ya en hora de Colombia (UTC-5)
    for c in ['fuente', 'confianza', 'dia_noche']:
        df[c] = df[c].str.strip().astype('category')
    print(f'Leído {ruta.name}: {len(df):,} filas en {time.time() - t0:.1f} s (codificación {enc})')
    return df


focos_raw = leer_focos(RAW_FILE)
f = focos_raw
print(f'\nRango de fechas: {f.fecha.min()}  a  {f.fecha.max()}  (sin fecha: {f.fecha.isna().sum()})')
print('\nFocos por año y fuente:')
display(pd.crosstab(f.fecha.dt.year, f.fuente, margins=True, margins_name='Total'))
es_pct = f.confianza.astype(str).str.contains('%')
print('Confianza categórica (VIIRS):')
display(f.loc[~es_pct, 'confianza'].astype(str).value_counts().to_frame())
print('Confianza en porcentaje (MODIS), por fuente:')
display(pd.crosstab(f.fuente, es_pct.map({True: 'porcentaje', False: 'categórica'})))
print('Día / noche:')
display(f.dia_noche.value_counts().to_frame())
print('Nulos por columna:')
display(f.isna().sum().to_frame('nulos'))
display(f[['lat', 'lon', 'frp', 'scan_km', 'track_km']].describe().round(3))
fuera = ~(f.lat.between(BBOX['lat_min'], BBOX['lat_max']) & f.lon.between(BBOX['lon_min'], BBOX['lon_max']))
print(f'Duplicados exactos: {f.duplicated().sum():,} | Fuera del rectángulo de Colombia: {fuera.sum():,}')
print('\nFocos por mes (últimos 6):')
display(f.groupby(f.fecha.dt.to_period('M')).size().tail(6).to_frame('focos'))

## Paso 1. Limpieza

In [ ]:
ORDEN_CONF = {'Baja': 0, 'Nominal': 1, 'Alta': 2}


def norm_conf(v):
    """Normaliza la confianza a Baja/Nominal/Alta (VIIRS categórica o MODIS en %)."""
    v = str(v).strip()
    if v in ('Alta', 'High'): return 'Alta'
    if v in ('Baja', 'Low'): return 'Baja'
    if v in ('Nominal', 'Medium'): return 'Nominal'
    try:
        pct = int(v.replace('%', '').strip())
        return 'Alta' if pct >= 65 else ('Nominal' if pct >= 30 else 'Baja')
    except ValueError:
        return 'Nominal'


def ultimo_mes_completo(fechas):
    """Último día del último mes calendario completo en la serie."""
    ultimo = fechas.max().normalize()
    fin_mes = ultimo + pd.offsets.MonthEnd(0)
    return ultimo if ultimo == fin_mes else ultimo.replace(day=1) - pd.Timedelta(days=1)


def limpiar_focos(df):
    """Aplica duplicados, periodo, fuentes, rectángulo de Colombia y confianza mínima, informando cada filtro."""
    t0 = time.time()
    df = df.copy()
    df['confianza'] = df['confianza'].astype(str).map(norm_conf)
    pasos = [
        ('duplicados exactos', lambda d: ~d.duplicated()),
        ('fuera del periodo', lambda d: d.fecha.between(FECHA_INI, FECHA_FIN + pd.Timedelta(days=1), inclusive='left')),
        ('fuentes no VIIRS', lambda d: d.fuente.isin(FUENTES)),
        ('fuera de Colombia', lambda d: d.lat.between(BBOX['lat_min'], BBOX['lat_max'])
                                        & d.lon.between(BBOX['lon_min'], BBOX['lon_max'])),
        (f'confianza menor que {CONFIANZA_MIN}', lambda d: d.confianza.map(ORDEN_CONF) >= ORDEN_CONF[CONFIANZA_MIN]),
    ]
    for nombre, filtro in pasos:
        n0 = len(df)
        df = df[filtro(df)]
        print(f'  - {nombre}: {n0 - len(df):,} eliminados -> {len(df):,}')
    df['fuente'] = df['fuente'].cat.remove_unused_categories()
    df['confianza'] = df['confianza'].astype('category')
    df['fecha_dia'] = df['fecha'].dt.normalize()
    df['anio'] = df['fecha'].dt.year.astype('int16')
    df['mes'] = df['fecha'].dt.month.astype('int8')
    # índice de celda de la grilla (redondeo hacia abajo)
    df['celda_i'] = np.floor(df['lat'] / CELDA_GRADOS).astype('int32')
    df['celda_j'] = np.floor(df['lon'] / CELDA_GRADOS).astype('int32')
    print(f'Limpieza: {len(df):,} focos en {time.time() - t0:.1f} s')
    return df.reset_index(drop=True)


if FECHA_FIN is None:
    FECHA_FIN = ultimo_mes_completo(focos_raw['fecha'])
print(f'Periodo de análisis: {FECHA_INI:%Y-%m-%d} a {FECHA_FIN:%Y-%m-%d}')
focos = limpiar_focos(focos_raw)
display(pd.crosstab(focos.anio, focos.fuente, margins=True, margins_name='Total'))

try:
    focos.to_parquet(SALIDAS_IPI / 'focos_limpios.parquet', index=False)
    print('Copia limpia guardada en', SALIDAS_IPI / 'focos_limpios.parquet')
except ImportError:
    print('AVISO: falta pyarrow; no se guardó el parquet (pip install pyarrow)')

## Paso 2. Fuentes de calor persistentes
Celdas de 0.004° con detecciones en todos los años, muchos días por año y casi todos los meses: típicamente teas de gas o industria. Se marcan junto con sus 8 celdas vecinas y esos focos se excluyen del índice.

In [ ]:
def estadisticas_celdas(df):
    """Resume por celda de la grilla los años, días y meses con detección y el FRP mediano."""
    dias = df.drop_duplicates(['celda_i', 'celda_j', 'fecha_dia'])
    por_anio = dias.groupby(['celda_i', 'celda_j', 'anio']).size()
    est = pd.DataFrame({
        'n_anios_activos': por_anio.groupby(level=[0, 1]).size(),
        'mediana_dias_por_anio': por_anio.groupby(level=[0, 1]).median(),
        'n_meses_calendario': dias.groupby(['celda_i', 'celda_j'])['mes'].nunique(),
        'total_dias': dias.groupby(['celda_i', 'celda_j']).size(),
        'frp_mediana': df.groupby(['celda_i', 'celda_j'])['frp'].median(),
    }).reset_index()
    est['lat_centro'] = (est.celda_i + 0.5) * CELDA_GRADOS
    est['lon_centro'] = (est.celda_j + 0.5) * CELDA_GRADOS
    return est


def ampliar_vecinas(celdas):
    """Devuelve el conjunto de celdas (i, j) junto con sus 8 vecinas."""
    return {(i + di, j + dj) for i, j in celdas for di in (-1, 0, 1) for dj in (-1, 0, 1)}


t0 = time.time()
celdas = estadisticas_celdas(focos)
celdas['persistente'] = ((celdas.n_anios_activos >= MIN_ANIOS)
                         & (celdas.mediana_dias_por_anio >= MIN_DIAS_ANIO)
                         & (celdas.n_meses_calendario >= MIN_MESES))
persistentes = celdas[celdas.persistente].copy()
marcadas = ampliar_vecinas(zip(persistentes.celda_i, persistentes.celda_j))
focos['excluido_persistente'] = pd.MultiIndex.from_arrays([focos.celda_i, focos.celda_j]).isin(list(marcadas))

print(f'Celdas con detección: {len(celdas):,} | persistentes: {len(persistentes):,} '
      f'| marcadas con vecinas: {len(marcadas):,}  ({time.time() - t0:.1f} s)')
print(f'Focos excluidos: {focos.excluido_persistente.sum():,} de {len(focos):,} '
      f'({focos.excluido_persistente.mean():.1%})')

## Paso 3. Asignación a municipio
Se ejecuta antes del mapa y la tabla del Paso 2, que necesitan el municipio de cada foco excluido. Cruce punto en polígono con los límites municipales (sin buffers).

In [ ]:
def cargar_municipios():
    """Carga polígonos municipales y lista maestra; reporta códigos que no coinciden y calcula el área en km²."""
    poli = gpd.read_file(WEB_DATA / 'municipios_colombia_geo.gpkg')
    poli['COD_MPIO'] = pd.to_numeric(poli['COD_MPIO']).astype(int)
    poli['area_km2'] = poli.to_crs(epsg=9377).area / 1e6
    maestra = pd.read_csv(WEB_DATA / 'amenazas_municipal.csv', encoding='utf-8-sig',
                          usecols=['COD_DPTO', 'NOM_DPTO', 'cod_divipola', 'NOM_MPIO', 'LATITUD', 'LONGITUD'])
    maestra = maestra.rename(columns={'cod_divipola': 'COD_MPIO'})
    solo_poli = sorted(set(poli.COD_MPIO) - set(maestra.COD_MPIO))
    solo_maestra = maestra[~maestra.COD_MPIO.isin(poli.COD_MPIO)]
    print(f'Polígonos: {len(poli):,} | lista maestra: {len(maestra):,}')
    print(f'  En polígonos y no en la lista maestra ({len(solo_poli)}): {solo_poli}')
    print(f'  En la lista maestra y sin polígono ({len(solo_maestra)}): '
          f'{solo_maestra[["COD_MPIO", "NOM_MPIO", "NOM_DPTO"]].values.tolist()}')
    return poli[['COD_MPIO', 'area_km2', 'geometry']], maestra


def asignar_municipio(df, poli):
    """Asigna COD_MPIO a cada foco por cruce punto en polígono (predicado within)."""
    t0 = time.time()
    pts = gpd.GeoDataFrame(index=df.index, geometry=gpd.points_from_xy(df.lon, df.lat), crs='EPSG:4326')
    unido = gpd.sjoin(pts, poli[['COD_MPIO', 'geometry']], how='left', predicate='within')
    unido = unido[~unido.index.duplicated(keep='first')]
    cod = unido['COD_MPIO'].reindex(df.index)
    print(f'Cruce espacial: {len(df):,} focos en {time.time() - t0:.1f} s | sin municipio: {cod.isna().sum():,}')
    return cod.astype('Int32')


poli_mpio, maestra = cargar_municipios()
focos['COD_MPIO'] = asignar_municipio(focos, poli_mpio)

### Paso 2 (cont.). Mapa y tabla de fuentes persistentes

In [ ]:
nombres = maestra.set_index('COD_MPIO')[['NOM_MPIO', 'NOM_DPTO']]
excl = focos[focos.excluido_persistente].join(nombres, on='COD_MPIO')

por_dpto = excl.groupby('NOM_DPTO', dropna=False).size().sort_values(ascending=False).rename('focos_excluidos')
top_mpio = (excl.groupby(['NOM_DPTO', 'NOM_MPIO'], dropna=False).size()
            .sort_values(ascending=False).head(20).rename('focos_excluidos'))
por_dpto.to_csv(SALIDAS_IPI / 'excluidos_por_departamento.csv', encoding='utf-8-sig')
top_mpio.to_csv(SALIDAS_IPI / 'excluidos_top20_municipios.csv', encoding='utf-8-sig')
display(por_dpto.to_frame())
display(top_mpio.to_frame())

# Municipio de cada celda persistente (según su centro)
centros = gpd.GeoDataFrame(persistentes, crs='EPSG:4326',
                           geometry=gpd.points_from_xy(persistentes.lon_centro, persistentes.lat_centro))
persistentes['COD_MPIO'] = gpd.sjoin(centros, poli_mpio[['COD_MPIO', 'geometry']], how='left',
                                     predicate='within')['COD_MPIO'].groupby(level=0).first().astype('Int32')
persistentes = persistentes.join(nombres, on='COD_MPIO')

fig, ax = plt.subplots(figsize=(9, 11))
poli_mpio.boundary.plot(ax=ax, linewidth=0.2, color='#9a9a9a')
sc = ax.scatter(persistentes.lon_centro, persistentes.lat_centro, c=persistentes.mediana_dias_por_anio,
                s=12 + persistentes.total_dias / persistentes.total_dias.max() * 80,
                cmap='YlOrRd', edgecolor='black', linewidth=0.3, zorder=3)
plt.colorbar(sc, ax=ax, shrink=0.5, label='Mediana de días con detección por año')
ax.set_title(f'Fuentes de calor persistentes ({len(persistentes)} celdas de {CELDA_GRADOS}°)\n'
             f'VIIRS {FECHA_INI:%Y-%m} a {FECHA_FIN:%Y-%m}; tamaño = días totales con detección')
ax.set_xlabel('Longitud'); ax.set_ylabel('Latitud'); ax.set_aspect('equal')
fig.tight_layout()
fig.savefig(SALIDAS_IPI / 'mapa_fuentes_persistentes.png', dpi=200)
plt.show()

display(persistentes.sort_values('total_dias', ascending=False).head(25)
        [['lat_centro', 'lon_centro', 'NOM_MPIO', 'NOM_DPTO', 'n_anios_activos', 'mediana_dias_por_anio',
          'n_meses_calendario', 'total_dias', 'frp_mediana']].round(3))

## Exportar CSV para Dashboard web
Puntos de calor del último día del archivo, sin filtros (todas las fuentes y confianzas), con el mismo formato que leía la página: `fecha_hora, lat, lon, fuente, temp_c, confianza, captura, municipio, departamento, cod_mpio`. El municipio se asigna por cruce punto en polígono.

In [ ]:
def exportar_ultimo_dia(df, poli, maestra, ruta):
    """Escribe incendios_ultimo_dia.csv con los focos del último día del archivo crudo."""
    ultimo = df['fecha'].max().normalize()
    dia = df[df['fecha'].dt.normalize() == ultimo].copy()
    dia['cod_mpio'] = asignar_municipio(dia, poli)
    dia = dia.join(maestra.set_index('COD_MPIO')[['NOM_MPIO', 'NOM_DPTO']], on='cod_mpio')
    web = pd.DataFrame({
        'fecha_hora': dia['fecha'].dt.strftime('%Y-%m-%d %H:%M'),
        'lat': dia['lat'],
        'lon': dia['lon'],
        'fuente': dia['fuente'].astype(str),
        'temp_c': dia['temp_i4'].astype(float).round(1),
        'confianza': dia['confianza'].astype(str).map(norm_conf),
        'captura': dia['dia_noche'].astype(str),
        'municipio': dia['NOM_MPIO'],
        'departamento': dia['NOM_DPTO'],
        'cod_mpio': dia['cod_mpio'],
    }).dropna(subset=['lat', 'lon'])
    web.to_csv(ruta, index=False, encoding='utf-8-sig')
    print(f'Exportado: {ruta}')
    print(f'Puntos: {len(web):,}  Fecha: {ultimo.date()}  ({ruta.stat().st_size / 1024:.1f} KB)')
    return web


df_web = exportar_ultimo_dia(focos_raw, poli_mpio, maestra, WEB_CSV)
print(df_web['confianza'].value_counts().to_string())
print('\nPor departamento (top 10):')
print(df_web['departamento'].value_counts().head(10).to_string())